In [68]:
!dpkg -l | grep tensorrt

In [70]:
!pip install nvidia-tensorrt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.9 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-11.1.0.106-py3-none-any.whl size=16564 sha256=1b475c386f86ad091e0afa41899973dca4e5122134e6d25f49933e51eeeb896a
  Stored in directory: /root/.cache/pip/wheels/64/60/e6/9df6cdc73f3d2d55f99b57987892a29d470494bf419be000ad
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-11.1.0.106-py3-none-any.whl size=23074 sha256=06877c4de12dc9e5d05c622498a84f2fda867b7fcefc9df16e8a8ab310fe41c6
  Stored in directory: /root/.cache/pip/wheels/77/0f/83/d9c20d0e840bfaf19d384d48255

In [71]:
import tensorrt as trt
print(trt.__version__)

11.1.0.106


In [75]:
import tensorrt as trt

ONNX_FILE = "/content/rfdetr-seg-small.onnx"
ENGINE_FILE = "/content/model.engine"

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)


network = builder.create_network()

parser = trt.OnnxParser(network, logger)

with open(ONNX_FILE, "rb") as f:
    if not parser.parse(f.read()):
        print("❌ ONNX Parsing Failed")
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise SystemExit()
    else:
        print("✅ ONNX Parsed Successfully")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

print("⏳ Building engine (serialized)...")

# ✅ NEW API (THIS FIXES YOUR ERROR)
serialized_engine = builder.build_serialized_network(network, config)

if serialized_engine is None:
    raise RuntimeError("❌ Engine build failed")

# Save engine
with open(ENGINE_FILE, "wb") as f:
    f.write(serialized_engine)

print(" Engine saved:", ENGINE_FILE)

✅ ONNX Parsed Successfully
⏳ Building engine (serialized)...
✅ Engine saved: /content/model.engine
